# BitFlip — Qiskit Fall Fest 2026 Phase 1

This notebook is a reproducible scaffold for a weighted Max-Cut/QAOA processor A/B comparison. **Before submission, verify the official problem instance and Processor B coupling map.** The currently included Processor B model is explicitly provisional.

## 1. Environment and processor definitions

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from src.problem import EDGES, NUM_VERTICES, exact_max_cut, cut_value
from src.processors import load_processor
from src.metrics import circuit_metrics

ROOT = Path.cwd()
if not (ROOT / "processors").exists():
    ROOT = Path(".")
processor_A = load_processor(ROOT / "processors/processor_A.json")
processor_B = load_processor(ROOT / "processors/processor_B.json")
print(processor_A["name"], processor_A["num_qubits"], "qubits")
print(processor_B["name"], processor_B["num_qubits"], "qubits")
print("WARNING:", processor_B["definition_status"])

## 2. Weighted Max-Cut instance and exact classical baseline

The graph below is the team’s working test instance. Replace it with the organizer-provided instance if one is specified in the Challenge Kit.

In [ ]:
print("Edges (u, v, weight):", EDGES)
baseline = exact_max_cut()
print("Exact classical baseline:", baseline)

## 3. Build a p=1 QAOA circuit

The circuit uses a standard cost unitary for weighted Max-Cut and an X-mixer. Bitstrings are evaluated against the weighted graph. The parameter values below are example values; for a final report, document the parameter-selection method and use identical settings for A and B.

In [ ]:
def build_qaoa_p1(gamma: float, beta: float, edges=EDGES, n=NUM_VERTICES):
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    # For Max-Cut, exp(-i gamma * w/2 * (1 - ZiZj)) is equivalent
    # up to a global phase to an RZZ interaction plus a phase.
    for u, v, w in edges:
        qc.rzz(-gamma * w, u, v)
    for q in range(n):
        qc.rx(2 * beta, q)
    qc.measure(range(n), range(n))
    return qc

gamma, beta = 0.25556, 1.18889
logical_circuit = build_qaoa_p1(gamma, beta)
logical_circuit.draw("mpl")

## 4. Compile against Processor A and Processor B

These metrics are compiler/resource measurements, not hardware results. The processor noise dictionaries are illustrative and are not automatically applied by transpilation.

In [ ]:
compiled = {}
rows = []
for label, processor in [("A", processor_A), ("B", processor_B)]:
    qc = transpile(
        logical_circuit,
        basis_gates=processor.get("basis_gates"),
        coupling_map=processor["coupling_map"],
        optimization_level=3,
        seed_transpiler=42,
    )
    compiled[label] = qc
    m = circuit_metrics(qc)
    rows.append({"processor": label, **{k:v for k,v in m.items() if k != "operations"}})
    print(f"Processor {label}")
    print(m)
architecture_table = pd.DataFrame(rows)
architecture_table

## 5. Optional simulator run

This cell runs the compiled circuits on an ideal Aer simulator as a plumbing check. It does not model the illustrative noise metadata and must not be described as a real-hardware result.

In [ ]:
sim = AerSimulator()
ideal_rows = []
for label, qc in compiled.items():
    result = sim.run(qc, shots=4096, seed_simulator=42).result()
    counts = result.get_counts()
    total = sum(counts.values())
    expected_cut = 0.0
    best_sample = 0
    for bitstring, count in counts.items():
        # Qiskit count strings are displayed most-significant classical bit first;
        # reverse to map character positions to qubit indices.
        bits = tuple(int(x) for x in bitstring[::-1])
        value = cut_value(bits)
        expected_cut += value * count / total
        best_sample = max(best_sample, value)
    ideal_rows.append({"processor": label, "shots": total, "expected_cut": expected_cut, "best_sampled_cut": best_sample})
ideal_results = pd.DataFrame(ideal_rows)
ideal_results

## 6. Save reproducible outputs

Review the outputs and ensure the Processor B definition is verified before treating comparisons as final.

In [ ]:
tables = ROOT / "results" / "tables"
tables.mkdir(parents=True, exist_ok=True)
architecture_table.to_csv(tables / "architecture_metrics.csv", index=False)
ideal_results.to_csv(tables / "ideal_simulator_results.csv", index=False)
print("Saved tables to", tables.resolve())